## **Exercise 04 : A/B-testing**

### **04.1 Using only one query for each of the groups, create two dataframes: test_results and control_results with the columns time and avg_diff and only two rows**

In [1]:
import pandas as pd
import sqlite3
# create a connection to the database using the library sqlite3
connection = sqlite3.connect('../../data/checking-logs.sqlite')

# time should have the values: after and before
# avg_diff contains the average delta among all the users for the time period before each of them made their first visit to the page and afterward
# only take into account the users that have observations before and after
# we still are not using the lab ’project1’
test_results = pd.read_sql('with users as (select uid, labname as labs, first_commit_ts, first_view_ts from test group by uid, labname)\
                        \
                        select "before" as time, avg(strftime("%s", first_commit_ts) - deadlines) / (60*60) as avg_diff \
                        from users left join deadlines using(labs) where labs like "%laba%" and first_commit_ts < first_view_ts\
                        \
                        union all\
                        \
                        select "after" as time, avg(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as avg_diff \
                        from users left join deadlines using(labs) where labs like "%laba%" and first_commit_ts > first_view_ts', connection)
test_results

,time,avg_diff
0,before,-61.156632
1,after,-103.953446


In [2]:
control_results = pd.read_sql('with users as (select uid, labname as labs, first_commit_ts, first_view_ts from control group by uid, labname)\
                        \
                        select "before" as time, avg(strftime("%s", first_commit_ts) - deadlines) / (60*60) as avg_diff \
                        from users left join deadlines using(labs) where labs like "%laba%" and first_commit_ts < first_view_ts\
                        \
                        union all\
                        \
                        select "after" as time, avg(strftime("%s", first_commit_ts) - deadlines)/ (60*60) as avg_diff \
                        from users left join deadlines using(labs) where labs like "%laba%" and first_commit_ts > first_view_ts', connection)
control_results

,time,avg_diff
0,before,-99.901448
1,after,-113.232346


In [3]:
connection.close()

In [4]:
# have the answer: did the hypothesis turn out to be true and the page does affect the students’ behavior?
round(float(abs((test_results['avg_diff'][0] - test_results['avg_diff'][1])/ test_results['avg_diff'][0])),3)

0.7

In test group after visiting the page users made first commit 70% faster

In [5]:
round(float(abs((control_results['avg_diff'][0] - control_results['avg_diff'][1])/ control_results['avg_diff'][0])),3)

0.133

In test group after visiting the page users made first commit 13% faster